In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/cleaned_data.csv')
print("Shape:", df.shape)

Shape: (404800, 27)


In [2]:
df['total_monthly_expenses'] = (
    df['monthly_rent'] +
    df['school_fees'] +
    df['college_fees'] +
    df['travel_expenses'] +
    df['groceries_utilities'] +
    df['other_monthly_expenses'] +
    df['current_emi_amount']
)

print(df['total_monthly_expenses'].describe())

count    404800.000000
mean      44639.560099
std       23072.348857
min        3600.000000
25%       27900.000000
50%       40600.000000
75%       56700.000000
max      247800.000000
Name: total_monthly_expenses, dtype: float64


In [3]:
df['disposable_income'] = df['monthly_salary'] - df['total_monthly_expenses']

print(df['disposable_income'].describe())
print(f"\nNegative disposable income (spending more than earning): {(df['disposable_income'] < 0).sum():,}")

count    404800.000000
mean      14831.331744
std       37581.517747
min     -161883.000000
25%         400.000000
50%       10600.000000
75%       22400.000000
max      485540.000000
Name: disposable_income, dtype: float64

Negative disposable income (spending more than earning): 97,618


In [4]:
df['expense_to_income_ratio'] = df['total_monthly_expenses'] / (df['monthly_salary'] + 1)

print(df['expense_to_income_ratio'].describe())


count    404800.000000
mean          0.867160
std           0.636453
min           0.011144
25%           0.606754
50%           0.779105
75%           0.991181
max          31.285799
Name: expense_to_income_ratio, dtype: float64


In [5]:
df['debt_to_income_ratio'] = df['current_emi_amount'] / (df['monthly_salary'] + 1)

print(df['debt_to_income_ratio'].describe())

count    404800.000000
mean          0.083591
std           0.142251
min           0.000000
25%           0.000000
50%           0.000000
75%           0.174420
max           7.702943
Name: debt_to_income_ratio, dtype: float64


In [6]:
# Combined savings
df['total_savings'] = df['bank_balance'] + df['emergency_fund']

# How many months of expenses covered by savings
df['savings_to_expense_ratio'] = df['total_savings'] / (df['total_monthly_expenses'] + 1)

# Requested EMI as % of disposable income
df['requested_emi'] = df['requested_amount'] / (df['requested_tenure'] + 1)
df['emi_to_disposable_ratio'] = df['requested_emi'] / (df['disposable_income'] + 1)

print(df[['total_savings', 'savings_to_expense_ratio', 
          'requested_emi', 'emi_to_disposable_ratio']].describe())

       total_savings  savings_to_expense_ratio  requested_emi  \
count   4.048000e+05             404800.000000  404800.000000   
mean    3.378044e+05                  8.168120   13850.482552   
std     2.573780e+05                  5.392407   12760.724368   
min     8.100000e+03                  0.187496     400.000000   
25%     1.469000e+05                  3.997332    5458.333333   
50%     2.729000e+05                  7.201032   10000.000000   
75%     4.603000e+05                 11.158728   17850.000000   
max     2.453600e+06                 49.755871  115000.000000   

       emi_to_disposable_ratio  
count            404800.000000  
mean                 30.091734  
std                 892.554904  
min               -3088.888889  
25%                   0.031300  
50%                   0.402461  
75%                   1.199869  
max               96800.000000  


In [7]:
# 1 if person has existing loans, 0 if not
df['has_existing_loan'] = (df['existing_loans'] == 'Yes').astype(int)

# 1 if disposable income is negative
df['is_financially_stressed'] = (df['disposable_income'] < 0).astype(int)

# 1 if credit score is good (above 700)
df['good_credit'] = (df['credit_score'] >= 700).astype(int)

print("Has existing loan:", df['has_existing_loan'].value_counts().to_dict())
print("Financially stressed:", df['is_financially_stressed'].value_counts().to_dict())
print("Good credit:", df['good_credit'].value_counts().to_dict())

Has existing loan: {0: 243227, 1: 161573}
Financially stressed: {0: 307182, 1: 97618}
Good credit: {1: 206278, 0: 198522}


In [8]:
# Label encode binary columns
binary_cols = {
    'gender': {'Male': 1, 'Female': 0},
    'marital_status': {'Married': 1, 'Single': 0},
    'existing_loans': {'Yes': 1, 'No': 0}
}

for col, mapping in binary_cols.items():
    df[col] = df[col].map(mapping)

print("Binary columns encoded!")
print(df[['gender', 'marital_status', 'existing_loans']].head())

Binary columns encoded!
   gender  marital_status  existing_loans
0       0               1               1
1       0               1               1
2       1               1               0
3       0               1               0
4       0               1               0


In [10]:
# One-hot encode multi-category columns
multi_cols = ['education', 'employment_type', 'company_type', 
              'house_type', 'emi_scenario']

df = pd.get_dummies(df, columns=multi_cols, drop_first=True)

print("Shape after encoding:", df.shape)
print("New columns added:")
print([col for col in df.columns if any(c in col for c in multi_cols)])

Shape after encoding: (404800, 48)
New columns added:
['education_High School', 'education_Post Graduate', 'education_Professional', 'employment_type_Private', 'employment_type_Self-employed', 'company_type_MNC', 'company_type_Mid-size', 'company_type_Small', 'company_type_Startup', 'house_type_Own', 'house_type_Rented', 'emi_scenario_Education EMI', 'emi_scenario_Home Appliances EMI', 'emi_scenario_Personal Loan EMI', 'emi_scenario_Vehicle EMI']


In [16]:
df.head()

,age,gender,marital_status,monthly_salary,years_of_employment,monthly_rent,family_size,dependents,school_fees,college_fees,...,company_type_MNC,company_type_Mid-size,company_type_Small,company_type_Startup,house_type_Own,house_type_Rented,emi_scenario_Education EMI,emi_scenario_Home Appliances EMI,emi_scenario_Personal Loan EMI,emi_scenario_Vehicle EMI
0,38,0,1,82600.0,0.9,20000.0,3,2,0,0,...,False,True,False,False,False,True,False,False,True,False
1,38,0,1,21500.0,7.0,0.0,2,1,5100,0,...,True,False,False,False,False,False,False,False,False,False
2,38,1,1,86100.0,5.8,0.0,4,3,0,0,...,False,False,False,True,True,False,True,False,False,False
3,58,0,1,66800.0,2.2,0.0,5,4,11400,0,...,False,True,False,False,True,False,False,False,False,True
4,48,0,1,57300.0,3.4,0.0,4,3,9400,21300,...,False,True,False,False,False,False,False,True,False,False


In [17]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['emi_eligibility_encoded'] = le.fit_transform(df['emi_eligibility'])

print("Class mapping:")
for i, cls in enumerate(le.classes_):
    print(f"  {cls} → {i}")

Class mapping:
  Eligible → 0
  High_Risk → 1
  Not_Eligible → 2


In [18]:
df.to_csv('data/engineered_data.csv', index=False)
print("Saved! Shape:", df.shape)
print("Columns:", df.columns.tolist())

Saved! Shape: (404800, 49)
Columns: ['age', 'gender', 'marital_status', 'monthly_salary', 'years_of_employment', 'monthly_rent', 'family_size', 'dependents', 'school_fees', 'college_fees', 'travel_expenses', 'groceries_utilities', 'other_monthly_expenses', 'existing_loans', 'current_emi_amount', 'credit_score', 'bank_balance', 'emergency_fund', 'requested_amount', 'requested_tenure', 'emi_eligibility', 'max_monthly_emi', 'total_monthly_expenses', 'disposable_income', 'expense_to_income_ratio', 'debt_to_income_ratio', 'total_savings', 'savings_to_expense_ratio', 'requested_emi', 'emi_to_disposable_ratio', 'has_existing_loan', 'is_financially_stressed', 'good_credit', 'education_High School', 'education_Post Graduate', 'education_Professional', 'employment_type_Private', 'employment_type_Self-employed', 'company_type_MNC', 'company_type_Mid-size', 'company_type_Small', 'company_type_Startup', 'house_type_Own', 'house_type_Rented', 'emi_scenario_Education EMI', 'emi_scenario_Home Applianc